# HW2 — Profile & Optimize an Autoregressive Decode Loop

**Lecture mapping:** L1 §07 (Profiling) · L2 §01 (prefill/decode, KV cache) · L2 §03 (engine optimizations)

The harness below defines a deliberately slow greedy-decode loop (**V0**): no KV
cache (it recomputes the whole sequence every step), fp32 eager, and a host sync
every step. Profile it, find the bottlenecks, and write a fast, numerically
identical replacement.

## What you implement

| Part | Function |
|------|----------|
| 1 | `profile` — wrap a loop in `torch.profiler`, print a table, export a Chrome trace |
| 2 | `optimized_loop` — fast greedy decode, same tokens as V0 (fp32) |
| 3 | `generate_optimized` — build, warm up, and time your optimized generation |
| 4 | Writeup Q1–Q4 |

## Speedup targets

`optimized_loop` end-to-end vs the V0 baseline:

| Speedup vs V0 | |
|---------------|--|
| ≥ 4.0× | excellent |
| ≥ 3.0× | good |
| ≥ 1.5× | some speedup |
| < 1.5× | little to no speedup |

The speedup only counts if `optimized_loop` passes the fp32 correctness check and
the timed run is on a GPU.

## Rules

- PyTorch + `requirements.txt` only — no vLLM / TensorRT-LLM / SGLang.
- Don't edit the harness cell.
- `optimized_loop` must reproduce the baseline's greedy tokens exactly on the same
  fp32 model. `generate_optimized` may switch to half precision (bf16, or fp16 on Turing/T4) for the timed run.

## Where to look

- **KV cache** (L2 §01) — the baseline throws it away every step. Biggest win.
- **Host syncs** (L1 §07) — `.item()` on the critical path serializes CPU↔GPU.
- **`torch.compile` / CUDA graphs** (L2 §03) — fuse ops, cut launch overhead.
- **dtype** — half precision for the timed run (bf16, or fp16 on a T4 — small win on a model this size).

## Cell types

- **DO NOT EDIT** — fixed harness.
- **YOUR IMPLEMENTATION** — replace `raise NotImplementedError`.
- **SELF-CHECK** — asserts that must pass.
- **WRITEUP** — answer in the markdown cell.

Run top-to-bottom on the GPU. Submit the executed notebook plus the files under
`results/`.

## The fixed harness (DO NOT EDIT)

Tiny 2-layer Llama, the V0 baseline, the correctness check, the timing helper,
and the speedup targets.

In [22]:
# DO NOT EDIT — editing this cell invalidates your speedup numbers.
import os
import time

import torch
from transformers import LlamaConfig, LlamaForCausalLM

RESULTS_DIR = os.path.join("results", "hw2")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 0
PROMPT_LEN = 64
MAX_NEW_TOKENS = 128

# Speedup targets (optimized end-to-end vs V0 baseline).
TIERS = [
    (4.0, ">= 4.0x  (excellent)"),
    (3.0, ">= 3.0x  (good)"),
    (1.5, ">= 1.5x  (some speedup)"),
    (0.0, "< 1.5x  (little to no speedup)"),
]


def device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def _config() -> LlamaConfig:
    # Small enough to iterate fast, real enough to profile meaningfully.
    return LlamaConfig(
        vocab_size=32000,
        hidden_size=512,
        intermediate_size=1376,
        num_hidden_layers=2,
        num_attention_heads=8,
        num_key_value_heads=8,
        max_position_embeddings=4096,
    )


def build_model_and_input(dtype: torch.dtype = torch.float32):
    """Return (model, input_ids) on the active device with fixed random weights.

    The same SEED is used every call, so the V0 baseline and your optimized run
    see identical weights and the same prompt.
    """
    torch.manual_seed(SEED)
    model = LlamaForCausalLM(_config()).to(device=device(), dtype=dtype).eval()
    input_ids = torch.randint(0, 32000, (1, PROMPT_LEN), device=device())
    return model, input_ids


@torch.no_grad()
def baseline_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    """V0 — intentionally slow. Greedy decode with NO KV cache: every step
    re-runs the model over the whole growing sequence (O(n^2)), and forces a
    host sync each step (the `.item()`) — exactly the anti-patterns from lecture.

    Returns the generated token ids, shape (1, max_new_tokens).
    """
    generated = input_ids
    out_tokens = []
    for _ in range(max_new_tokens):
        out = model(input_ids=generated, use_cache=False)
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        _ = next_tok.item()  # forced host sync every step
        generated = torch.cat([generated, next_tok], dim=1)
        out_tokens.append(next_tok)
    return torch.cat(out_tokens, dim=1)


def time_loop(loop_fn, *args, n_warmup: int = 1, n_iters: int = 3) -> float:
    """Average seconds per full generation of `loop_fn(*args)`."""
    for _ in range(n_warmup):
        loop_fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(n_iters):
            loop_fn(*args)
        end.record()
        torch.cuda.synchronize()
        return (start.elapsed_time(end) / 1e3) / n_iters
    t0 = time.perf_counter()
    for _ in range(n_iters):
        loop_fn(*args)
    return (time.perf_counter() - t0) / n_iters


def check_correctness(reference: torch.Tensor, candidate: torch.Tensor) -> bool:
    """Greedy decoding is deterministic, so a correct optimized loop must
    reproduce the baseline's token ids EXACTLY when given the same fp32 model."""
    return reference.shape == candidate.shape and torch.equal(
        reference.cpu(), candidate.cpu()
    )


def speedup_tier(speedup: float) -> str:
    for threshold, label in TIERS:
        if speedup >= threshold:
            return label
    return TIERS[-1][1]


print(f"device={device()}  prompt_len={PROMPT_LEN}  new_tokens={MAX_NEW_TOKENS}")

device=cuda  prompt_len=64  new_tokens=128


## Part 1 — profile a generation loop

See the L1 §07 `torch.profiler` snippet. You will use this on both the baseline
and your optimized loop, and read the traces at
[ui.perfetto.dev](https://ui.perfetto.dev).

In [23]:
# TODO-CELL: profile
# YOUR IMPLEMENTATION
def profile(loop_fn, model, input_ids, max_new_tokens: int, trace_path: str) -> None:
    """Run `loop_fn(model, input_ids, max_new_tokens)` under torch.profiler.

    Requirements:
      * Profile both CPU and CUDA activity (ProfilerActivity).
      * Print a short table sorted by self CUDA time (key_averages().table(...))
        — on CPU-only machines sort by self CPU time instead.
      * Export a Chrome trace to `trace_path`.
    """
    activities = [torch.profiler.ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(torch.profiler.ProfilerActivity.CUDA)

    trace_dir = os.path.dirname(trace_path)
    if trace_dir:
        os.makedirs(trace_dir, exist_ok=True)

    with torch.profiler.profile(activities=activities) as prof:
        loop_fn(model, input_ids, max_new_tokens)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    sort_by = (
        "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    )
    print(prof.key_averages().table(sort_by=sort_by, row_limit=10))
    prof.export_chrome_trace(trace_path)

## Part 2 — a fast, correct greedy decode loop

Return the same token ids as `baseline_loop` for the same fp32 model — shape
`(1, max_new_tokens)`. Start with what the baseline recomputes every step and
what it forces onto the host.

In [24]:
# TODO-CELL: optimized_loop
# YOUR IMPLEMENTATION
@torch.no_grad()
def optimized_loop(model, input_ids, max_new_tokens: int) -> torch.Tensor:
    """Greedy decode, but fast. Same tokens as baseline_loop, much less work."""
    if max_new_tokens == 0:
        return input_ids.new_empty((input_ids.shape[0], 0))

    out_tokens = input_ids.new_empty((input_ids.shape[0], max_new_tokens))

    out = model(input_ids=input_ids, use_cache=True)
    next_tok = out.logits[:, -1:, :].argmax(dim=-1)
    past_key_values = out.past_key_values
    out_tokens[:, 0:1] = next_tok

    for step in range(1, max_new_tokens):
        out = model(
            input_ids=next_tok,
            past_key_values=past_key_values,
            use_cache=True,
        )
        next_tok = out.logits[:, -1:, :].argmax(dim=-1)
        past_key_values = out.past_key_values
        out_tokens[:, step : step + 1] = next_tok

    return out_tokens

## Part 3 — build + time the optimized generation

Build the model however you like (dtype, `torch.compile`, CUDA graphs), run
`optimized_loop`, and return `(elapsed_seconds, generated_token_ids)`.

`elapsed_seconds` must be a warmed-up, timed measurement (use `time_loop`); it's
what gets compared to V0. Numerics-changing tricks (e.g. bf16) are fine here —
correctness is checked separately on the fp32 path.

In [25]:
# TODO-CELL: generate_optimized
# YOUR IMPLEMENTATION
def generate_optimized(max_new_tokens: int = MAX_NEW_TOKENS):
    """Return (elapsed_seconds, generated_token_ids) for your fastest setup."""
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        dtype = torch.float32

    model, input_ids = build_model_and_input(dtype)
    elapsed_seconds = time_loop(optimized_loop, model, input_ids, max_new_tokens)
    generated_token_ids = optimized_loop(model, input_ids, max_new_tokens)
    return elapsed_seconds, generated_token_ids

## Self-check (small + fast, runs anywhere)

In [26]:
# SELF-CHECK — DO NOT EDIT.
_n = 24  # short, for speed
_model, _ids = build_model_and_input(torch.float32)

_ref = baseline_loop(_model, _ids, _n)
_cand = optimized_loop(_model, _ids, _n)
assert tuple(_cand.shape) == (1, _n), (
    f"expected shape (1, {_n}), got {tuple(_cand.shape)}"
)
assert check_correctness(_ref, _cand), (
    "optimized_loop must reproduce the baseline greedy tokens EXACTLY (fp32)"
)
print("optimized_loop matches baseline   PASS")

_trace = os.path.join(RESULTS_DIR, "trace_check.json")
if os.path.exists(_trace):
    os.remove(_trace)
profile(baseline_loop, _model, _ids, 4, _trace)
assert os.path.exists(_trace) and os.path.getsize(_trace) > 0, (
    "profile() must write a non-empty Chrome trace file"
)
print("profile() writes a Chrome trace   PASS")
print()
print("All checks passed")

optimized_loop matches baseline   PASS
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         9.52%       1.587ms        13.60%       2.268ms      37.799us       6.663ms        83.40%       6.663ms     111.056us            60  
                                  volta_sgemm_128x64_tn         0.00%       0.000us         0.00%       0.000us       0.000us       5.512ms        68.98%       5.512ms 

## The full run — baseline vs optimized (GPU)

Prints your speedup and writes the two traces for the writeup.

In [27]:
# DO NOT EDIT — the full run.
N_NEW = MAX_NEW_TOKENS if torch.cuda.is_available() else 32
if N_NEW != MAX_NEW_TOKENS:
    print(
        "CPU detected — trimmed to 32 new tokens as a smoke test. "
        "REPORTED NUMBERS MUST COME FROM A GPU RUN."
    )

# Baseline (V0), fp32 — also the correctness reference.
model_fp32, input_ids = build_model_and_input(torch.float32)
base_t = time_loop(baseline_loop, model_fp32, input_ids, N_NEW)
ref = baseline_loop(model_fp32, input_ids, N_NEW)
print(f"baseline V0: {base_t * 1e3:.1f} ms/generation")

# Correctness of your loop on the SAME fp32 model.
cand = optimized_loop(model_fp32, input_ids, N_NEW)
ok = check_correctness(ref, cand)
print(f"optimized_loop correctness (fp32): {'PASS' if ok else 'FAIL'}")

# Optimized end-to-end timing (your choice of dtype/compile/graphs).
opt_t, _ = generate_optimized(N_NEW)
speedup = base_t / opt_t
print(f"optimized:   {opt_t * 1e3:.1f} ms/generation")
print(f"speedup: {speedup:.2f}x  ->  {speedup_tier(speedup)}")
if not ok:
    print("WARNING: correctness FAILED — the speedup does not count.")

# Two traces for the writeup: baseline vs optimized.
profile(
    baseline_loop,
    model_fp32,
    input_ids,
    32,
    os.path.join(RESULTS_DIR, "trace_baseline.json"),
)
profile(
    optimized_loop,
    model_fp32,
    input_ids,
    32,
    os.path.join(RESULTS_DIR, "trace_optimized.json"),
)
print("wrote traces to results/hw2/ — open them at https://ui.perfetto.dev")

baseline V0: 605.0 ms/generation
optimized_loop correctness (fp32): PASS
optimized:   473.0 ms/generation
speedup: 1.28x  ->  < 1.5x  (little to no speedup)
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         9.06%      12.295ms        13.23%      17.951ms      37.398us      77.648ms        82.67%      77.648ms     161.768us           480  
                                  volta_sgemm_128x

---
## WRITEUP (be concrete and quantitative)

### Q1

From your **baseline trace**, what dominates the time? Name the specific symptom
you saw in the profiler (e.g. kernel-launch gaps, a long run of tiny kernels,
host syncs) and explain it.

**Your answer:**

> The baseline is mostly spending time in matrix multiplications. In my 32-token baseline profile, `aten::mm` took `77.648 ms`, which was `82.67%` of the self CUDA time. So the main symptom is that the baseline keeps doing a lot of GEMM work over and over.
>
> The reason is that V0 does not use a KV cache. Every time it generates one new token, it runs the model again on the whole growing sequence, not just the new token. It also calls `.item()` every step, which forces the CPU to wait for the GPU. In plain terms: the baseline repeats a lot of old work and keeps checking back with the CPU while decoding.

### Q2

List each optimization you applied and the speedup it contributed — measure them
**one at a time** (baseline → +KV cache → +compile → …). A small table is ideal.

**Your answer:**

> These are the numbers from my T4 GPU run. The final optimized run was correct in fp32, so the speedup counts, but it is still in the lowest tier.
>
> | Version | What changed | Time per generation | Speedup |
> |---|---|---:|---:|
> | Baseline V0 | No KV cache, fp32, `.item()` sync each step | `605.0 ms` | `1.00x` |
> | Final optimized | KV cache, no `.item()` sync, preallocated output tensor, fp16 for timed run | `473.0 ms` | `1.28x` |
>
> I also tried more advanced ideas like static cache and `torch.compile`, but on this small model/T4 setup they made the run slower, so I did not keep them. My best kept result was `1.28x`: better than baseline, but still reported by the harness as `< 1.5x (little to no speedup)`.

### Q3

Which single change had the biggest impact, and why does it help THIS workload
(128 short decode steps on a tiny model) specifically?

**Your answer:**

> The biggest useful change was using the KV cache. Without the cache, the model recalculates attention keys and values for the prompt plus all previously generated tokens at every step. With the cache, the old keys and values are saved, so each new step mostly handles only the newest token.
>
> This should help a lot for autoregressive decoding because generation is one-token-at-a-time. In my trace, the optimized 32-token profile had much less total GPU compute time than baseline: self CUDA time went from `93.923 ms` down to `26.243 ms`. The full end-to-end speedup was only `1.28x`, though, because this tiny model does many small GPU operations, and the overhead of launching lots of little kernels is still expensive on the T4.

### Q4

Your decode loop is memory-bandwidth-bound (L1 §06, L2 §01). Which of your
optimizations attack memory traffic vs CPU/launch overhead? Which kind mattered
more here, and why?

**Your answer:**

> The KV cache mainly attacks memory traffic and repeated compute. Instead of rereading and recomputing the whole sequence history every step, it keeps the old keys and values and reuses them. Using fp16 in the timed run also reduces memory traffic because the model weights and activations are smaller than fp32.
>
> Removing `.item()` attacks CPU/GPU synchronization overhead. Preallocating the output tensor is also a small CPU-side cleanup, because it avoids building a list of tiny tensors and concatenating them at the end.
>
> For this run, reducing repeated GPU work clearly helped in the profiler: self CUDA time dropped from `93.923 ms` in the baseline trace to `26.243 ms` in the optimized trace. But the final wall-clock speedup was only `1.28x`, so CPU/launch overhead still mattered a lot. My takeaway is: the cache reduced the actual GPU math, but this tiny decode workload is still slowed down by many small one-token steps.